This Jupyter Notebook creates a Folium map from the imported geojson file created earlier in the processing of the API dataset. First import the needed modules.

In [17]:
import folium
import geopandas as gpd
import json
import os
import panel as pn
from folium import IFrame
from PIL import Image
import base64
import webbrowser

Next we load the Panel extension and set the working directory. To initialize the map, we set the central latitude and longitude of the map and the zoom level of the initial visualization.  A higher zoom level (e.g., zoom level 18) means a closer, more detailed view of a smaller geographic area. This results in a larger scale, meaning that a smaller area is covered by the same number of pixels on the screen. We set the center of the map by defining the latitude and longitude of the centroid of the state and adjusting as needed for spacing and title block.  We are using a zoom level of 8, which is good for the extent of our dataset (the extent of the state of Kentucky).

In [18]:
# Load the Panel extension
pn.extension()

# Set working directory and initialize the map
cwd = os.getcwd()
m = folium.Map(location=[38, -85.5], zoom_start=8)

Load the county boundary GeoJSON file and extract the bounding boxes for zoom functionality.

In [19]:
# Load the GeoJSON data for counties
counties_gdf = gpd.read_file(f'{cwd}/data/reference_data/KY_Counties_WGS84.geojson')

# Extract the bounds for each county
county_bounds = {}
for _, row in counties_gdf.iterrows():
    name = row['NAME']  # Replace 'name' with the column containing the county names
    bounds = row['geometry'].bounds  # Get the bounding box as (minx, miny, maxx, maxy)
    county_bounds[name] = [[bounds[1], bounds[0]], [bounds[3], bounds[2]]]  # Convert to [[lat1, lon1], [lat2, lon2]]

# Convert bounds dictionary to JSON format
bounds_json = json.dumps(county_bounds)

Add custom JavaScript to display cursor location and county selector. This cell is not adding the desired functionality to the map but is not breaking things either!

In [20]:
# Add custom JavaScript to display cursor location and county selector
javascript_code = f"""
    <script>
    function updateLocation(e) {{
        var latlng = e.latlng;
        var lat = latlng.lat.toFixed(6);
        var lng = latlng.lng.toFixed(6);
        document.getElementById('location-info').innerHTML = 'Lat: ' + lat + ' | Lng: ' + lng;
    }}

    function zoomToCounty() {{
        var selectedCounty = document.getElementById('county-select').value;
        var bounds = boundsDict[selectedCounty];
        if (bounds) {{
            var latlngs = [
                [bounds[0][0], bounds[0][1]],
                [bounds[1][0], bounds[1][1]]
            ];
            var bounds = L.latLngBounds(latlngs);
            map.fitBounds(bounds);
        }}
    }}

    var map = L.map('map', {{
        center: [38.0, -85.5],
        zoom: 8
    }});

    map.on('mousemove', updateLocation);

    var infoControl = L.control({{position: 'bottomleft'}});
    infoControl.onAdd = function (map) {{
        this._div = L.DomUtil.create('div', 'info');
        this._div.id = 'location-info';
        this._div.innerHTML = 'Lat: -- | Lng: --';
        return this._div;
    }};
    infoControl.addTo(map);

    var selectorControl = L.control({{position: 'bottomright'}});
    selectorControl.onAdd = function (map) {{
        this._div = L.DomUtil.create('div', 'selector');
        this._div.innerHTML = `
            <label for="county-select">Select a county:</label>
            <select id="county-select" onchange="zoomToCounty()">
                <option value="">Select...</option>
                {''.join([f'<option value="{name}">{name}</option>' for name in county_bounds.keys()])}
            </select>
        `;
        return this._div;
    }};
    selectorControl.addTo(map);

    var boundsDict = {bounds_json};
    </script>
"""

# Add the JavaScript to the map
m.get_root().html.add_child(folium.Element(javascript_code))

While the following cell does not cause any errors, the desired element is not added to the map as intended.

In [21]:
# Add custom JavaScript to display cursor location
javascript_code = """
    <script>
    function updateLocation(e) {
        var latlng = e.latlng;
        var lat = latlng.lat.toFixed(6);
        var lng = latlng.lng.toFixed(6);
        document.getElementById('location-info').innerHTML = 'Lat: ' + lat + ' | Lng: ' + lng;
    }

    var map = L.map('map', {
        center: [38.0, -85.5],
        zoom: 8
    });

    map.on('mousemove', updateLocation);

    var infoControl = L.control({position: 'bottomleft'});
    infoControl.onAdd = function (map) {
        this._div = L.DomUtil.create('div', 'info');
        this._div.id = 'location-info';
        this._div.innerHTML = 'Lat: -- | Lng: --';
        return this._div;
    };
    infoControl.addTo(map);
    </script>
"""

# Add the JavaScript to the map
m.get_root().html.add_child(folium.Element(javascript_code))

# Save the map to an HTML file
m.save(f"{cwd}/web/KY_Incident_Locations.html")

Add custom CSS to control the size of the LayerControl

In [22]:
# Define custom CSS for LayerControl
custom_css = """
<style>
    .leaflet-control-layers {
        font-size: 12px; /* Adjust font size */
    }
    .leaflet-control-layers-toggle {
        width: 150px; /* Adjust width of the toggle button */
        height: 150px; /* Adjust height of the toggle button */
    }
    .leaflet-control-layers-list {
        max-height: 250px; /* Adjust the maximum height of the control list */
        overflow-y: auto; /* Add scrollbar if the list exceeds max-height */
    }
</style>
"""

# Add the CSS to the map
m.get_root().html.add_child(folium.Element(custom_css))

In [23]:
# Define the path to your north arrow image
north_arrow_image_path = f'{cwd}/data/reference_data/north_arrow.png'

# Encode the image in base64
with open(north_arrow_image_path, 'rb') as f:
    encoded_image = base64.b64encode(f.read()).decode()

# Create HTML content for the DivIcon
html = f'''
    <div style="
        background-image: url('data:image/png;base64,{encoded_image}');
        background-size: contain;
        background-repeat: no-repeat;
        width: 100px;  /* Adjust size as needed */
        height: 100px; /* Adjust size as needed */
        border: none;
    "></div>
'''

# Add the north arrow to the map
folium.Marker(
    location=[39.5, -82.5],  # Adjust the location to where you want the north arrow
    icon=folium.DivIcon(html=html),
).add_to(m)

The next cell allows for the addition of a different tile layer for the background.  It has not been implemented yet.

In [24]:
# Add a tile layer from a REST service
folium.TileLayer(
    tiles='https://stamen-tiles-{s}.a.ssl.fastly.net/toner/{z}/{x}/{y}.png',
    attr='Map tiles by <a href="https://stamen.com">Stamen Design</a>, under <a href="https://creativecommons.org/licenses/by/3.0">CC BY 3.0</a>. Data by <a href="https://openstreetmap.org">OpenStreetMap</a>, under <a href="https://www.openstreetmap.org/copyright">ODbL</a>.',
    name='Stamen Toner',
    control=False
).add_to(m)

Nex, we load the GeoJSON data from the data folders.

In [25]:
# Load the GeoJSON data from the data folder
with open(f'{cwd}/data/api_clean_data/Roadway_Characteristics_API.geojson') as f:
    roads_geojson = json.load(f)

# Load the GeoJSON data for counties and districts
with open(f'{cwd}/data/reference_data/KY_County_Polygons.geojson') as f:
    counties_geojson = json.load(f)

with open(f'{cwd}/data/reference_data/KYTC_Districts_Polygons.geojson') as f:
    districts_geojson = json.load(f)

The following function allows us to place markers on a map at specific locations, with each marker showing information about that location when clicked. The function goes through the list of locations in the GeoJSON, reading the latitude, longitude and the county name and route.  It creates a small pop-up box that shows this information. The pop-up will show the county name and route when you click on the marker. The function also sets the size, color and fill for the marker's symbology.

In [26]:
# Define a function to add simple point markers
def add_simple_marker(geojson, map_obj):
    for feature in geojson['features']:
        # Extract coordinates and properties
        coords = feature['geometry']['coordinates']
        props = feature['properties']

        # Create an HTML string for the popup
        popup_content = f"""
        <div style="width: 125px; height: 80px;">
            <p>County: {props.get('County_Name')}</p>
            <p>Route: {props.get('Route')}</p>
        </div>
        """
        iframe = IFrame(html=popup_content, width=125, height=60)
        popup = folium.Popup(iframe, max_width=150)


        # Create a simple circular marker
        folium.CircleMarker(
            location=[coords[1], coords[0]],
            radius=4,
            color='red',
            fill=True,
            fill_color='orange',
            fill_opacity=0.6,
            popup=popup
        ).add_to(map_obj)

In [27]:
# Define a function to add GeoJSON layers
def add_geojson_layer(geojson_data, layer_name, color, fill_color, opacity, fill_opacity, map_obj, show):
    folium.GeoJson(
        geojson_data,
        name=layer_name,
        show=show,
        style_function=lambda feature: {
            'fillColor': fill_color,
            'color': color,
            'weight': 1.2 if layer_name == "Districts" else 2,
            'opacity': opacity,
            'fillOpacity': fill_opacity
        }
    ).add_to(map_obj)

In [28]:
# Add GeoJSON layers
add_geojson_layer(counties_geojson, "Counties", "#E0D2B8", "#F3EDD3", 0.7, 0.3, m, True)
add_geojson_layer(districts_geojson, "Districts", "black", "none", 1, 0, m, True)

In [29]:
# Add a LayerControl to toggle visibility of the county and district boundary layers
folium.LayerControl().add_to(m)

The cell adds the markers to the map and saves the map to an HTML file.

In [30]:
# Add simple point markers to the map
add_simple_marker(roads_geojson, m)

# Save the map to an HTML file
m.save(f"{cwd}/web/KY_Incident_Locations.html")

In [31]:
# HTML content for the title block
title_html = '''
<!DOCTYPE html>
<html>
<head>
    <style>
        .title-block {
            position: absolute;
            top: 10px;
            left: 30%;
            transform: translateX(-50%);
            background-color: white;
            padding: 10px;
            border: 2px solid black;
            z-index: 1000;
        }
    </style>
</head>
<body>
    <div class="title-block">
        <h1>Collision Incidents within Construction Work Zones in Kentucky 2020-2024</h1>
        <p>** Data from Kentucky State Police Public Collision Data **</p>
    </div>
</body>
</body>
</html>
'''

# Read the existing HTML file
file_path = f'{cwd}/web/KY_Incident_Locations.html'
with open(file_path, 'r') as file:
    map_html = file.read()

# Insert the title block HTML into the map HTML
head_pos = map_html.find('<body>') + len('<body>')
map_html = map_html[:head_pos] + title_html + map_html[head_pos:]

# Write the modified HTML back to the file
with open(file_path, 'w') as file:
    file.write(map_html)

The html file is written to a specified folder location and then automatically opened in the default webbrowser.

In [32]:
# Open the HTML file in the default web browser
webbrowser.open(f'file://{file_path}')

True